In [1]:
RUN_TARGET = "colab"  # "colab" | "local"


## Setup Instructions

Notebook 07 generates the in silico mutagenesis CSV artifacts under `results/insilico_mutagenesis/`. Final plotting has been centralized in `08_compare_all_model_probes.ipynb` so this notebook can stay focused on data generation and reuse.

### Typical workflow
1. Reuse or regenerate the mutagenesis CSVs in this notebook.
2. Open `08_compare_all_model_probes.ipynb` to render the paper figures and the mutagenesis diagnostic plots from those saved CSVs.


# 07 - In Silico Mutagenesis


In [2]:
import ast
import importlib
import importlib.metadata as _md
import json
import subprocess as _sp
import sys
from pathlib import Path

# Allow imports from the src-layout package without a root-level shim.
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    _src_dir = _candidate / "src"
    if (_src_dir / "xallergen").exists() and str(_src_dir) not in sys.path:
        sys.path.insert(0, str(_src_dir))
        break

if RUN_TARGET == "colab":
    _required = {
        "numpy": "1.26.4",
        "pandas": "2.3.2",
        "scikit-learn": "1.8.0",
        "transformers": "4.48.1",
        "huggingface-hub": "0.36.2",
        "seaborn": "0.13.2",
        "matplotlib": "3.10.6",
    }

    def _version_matches(name: str, expected: str) -> bool:
        try:
            return _md.version(name) == expected
        except _md.PackageNotFoundError:
            return False

    _missing_or_mismatched = [
        f"{name}=={version}"
        for name, version in _required.items()
        if not _version_matches(name, version)
    ]

    if _missing_or_mismatched:
        print("Installing:", ", ".join(_missing_or_mismatched))
        _sp.run(
            [sys.executable, "-m", "pip", "install", "-q", "--upgrade", *_missing_or_mismatched],
            check=True,
        )
        raise SystemExit(
            "Colab environment updated. Restart the runtime once, then rerun the notebook from the top."
        )

    from google.colab import drive as _drive

    _drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/XAllergen2.0")
    DRIVE_MODELS = DRIVE_ROOT / "models"
    DRIVE_RESULTS = DRIVE_ROOT / "results"
    RUNTIME_ROOT = Path("/content/XAllergen2.0")
    if str(RUNTIME_ROOT) not in sys.path:
        sys.path.insert(0, str(RUNTIME_ROOT))
    _SRC_DIR = RUNTIME_ROOT / "src"
    if _SRC_DIR.exists() and str(_SRC_DIR) not in sys.path:
        sys.path.insert(0, str(_SRC_DIR))

import xallergen.baseline_notebook_utils as baseline_notebook_utils
importlib.reload(baseline_notebook_utils)
from xallergen.baseline_notebook_utils import *

if RUN_TARGET != "colab":
    configure_matplotlib_cache(Path.cwd())
    PROJECT_ROOT = find_project_root(Path.cwd())
    DEVICE = detect_device()
    print_runtime_context(DEVICE, PROJECT_ROOT)
    MODELS_DIR = PROJECT_ROOT / "models"
    RESULTS_DIR = PROJECT_ROOT / "results"
else:
    DRIVE_PROJECT_ROOT = DRIVE_ROOT if "DRIVE_ROOT" in globals() else Path("/content/drive/MyDrive/XAllergen2.0")
    PROJECT_ROOT = DRIVE_PROJECT_ROOT if DRIVE_PROJECT_ROOT.exists() else RUNTIME_ROOT
    DEVICE = detect_device()
    print_runtime_context(DEVICE, PROJECT_ROOT)
    MODELS_DIR = DRIVE_MODELS if DRIVE_MODELS.exists() else PROJECT_ROOT / "models"
    RESULTS_DIR = DRIVE_RESULTS if DRIVE_RESULTS.exists() else PROJECT_ROOT / "results"

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ISM_RESULTS_DIR = RESULTS_DIR / "insilico_mutagenesis"
ISM_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch


ICML_ONE_COLUMN_WIDTH = 4.2
PAPER_SHORT_HEIGHT = 3.0
PAPER_MEDIUM_HEIGHT = 3.6
PAPER_TALL_HEIGHT = 4.4
PAPER_DPI = 300
PAPER_TITLE_FONTSIZE = 13
PAPER_LABEL_FONTSIZE = 11.5
PAPER_TICK_FONTSIZE = 10.5
PAPER_LEGEND_FONTSIZE = 10.5
PAPER_LEGEND_TITLE_FONTSIZE = 11
PAPER_ANNOT_FONTSIZE = 10


def set_paper_plot_style() -> None:
    plt.rcParams.update({
        "font.size": PAPER_LABEL_FONTSIZE,
        "axes.titlesize": PAPER_TITLE_FONTSIZE,
        "axes.labelsize": PAPER_LABEL_FONTSIZE,
        "xtick.labelsize": PAPER_TICK_FONTSIZE,
        "ytick.labelsize": PAPER_TICK_FONTSIZE,
        "legend.fontsize": PAPER_LEGEND_FONTSIZE,
        "legend.title_fontsize": PAPER_LEGEND_TITLE_FONTSIZE,
        "figure.dpi": PAPER_DPI,
        "savefig.dpi": PAPER_DPI,
    })

BASELINE_CHECKPOINT_PATH = MODELS_DIR / "baseline_frozen_esm2.pt"
RAW_IG_ROWS_CANDIDATES = [
    RESULTS_DIR / "probing" / "rows" / "baseline_probing_rows.csv",
    RESULTS_DIR / "baseline_probing_rows.csv",
    RESULTS_DIR / "baseline_probe_rows.csv",
]
RAW_IG_ROWS_PATH = next((path for path in RAW_IG_ROWS_CANDIDATES if path.exists()), None)
if RAW_IG_ROWS_PATH is None:
    raise FileNotFoundError(
        "Could not find a baseline IG/probe rows artifact. Checked: "
        + ", ".join(str(path) for path in RAW_IG_ROWS_CANDIDATES)
    )

TEST_WITH_MASKS_CSV = DATA_DIR / "positives_splitB.csv"

model, baseline_checkpoint = load_baseline_checkpoint(BASELINE_CHECKPOINT_PATH, DEVICE)
tokenizer = build_tokenizer()

positive_test_df = pd.read_csv(TEST_WITH_MASKS_CSV).copy()
positive_test_df["sequence_id"] = positive_test_df.get("sequence_id", positive_test_df["accession"]).astype(str)
positive_test_df["epitope_label"] = [
    parse_epitope_label(seq, start, end)
    for seq, start, end in zip(
        positive_test_df["sequence"],
        positive_test_df["epitope_start"],
        positive_test_df["epitope_end"],
    )
]
positive_test_df["seq_len"] = positive_test_df["sequence"].str.len().astype(int)
positive_test_df["n_epitope_residues"] = positive_test_df["epitope_label"].map(lambda arr: int(np.asarray(arr).sum()))
positive_test_df["epitope_density"] = positive_test_df["n_epitope_residues"] / positive_test_df["seq_len"]
sequence_lookup_df = positive_test_df[["sequence_id", "accession", "sequence", "seq_len", "epitope_density"]].drop_duplicates()
sequence_lookup = sequence_lookup_df.set_index("sequence_id").to_dict(orient="index")

raw_ig_df = pd.read_csv(RAW_IG_ROWS_PATH)
print(f"Using baseline checkpoint: {BASELINE_CHECKPOINT_PATH}")
print(f"Using raw IG artifact: {RAW_IG_ROWS_PATH}")
print(f"Mutagenesis output directory: {ISM_RESULTS_DIR}")
print(f"Loaded splitB positives with epitope masks: {len(sequence_lookup_df)} proteins")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RUN_TARGET: local
Device: cuda
Project root: /content/drive/MyDrive/XAllergen2.0
GPU configuration:
  backend: CUDA
  CUDA available: True
  GPU count: 1
  Current device: NVIDIA A100-SXM4-40GB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/31.4M [00:00<?, ?B/s]

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using baseline checkpoint: /content/drive/MyDrive/XAllergen2.0/models/baseline_frozen_esm2.pt
Using raw IG artifact: /content/drive/MyDrive/XAllergen2.0/results/probing/rows/baseline_probing_rows.csv
Mutagenesis output directory: /content/drive/MyDrive/XAllergen2.0/results/insilico_mutagenesis
Loaded splitB positives with epitope masks: 61 proteins


## Analysis Parameters

- `K_PERCENTAGES` defines the top-k sweep as a fraction of sequence length, never as a fixed integer.
- `MIN_DELTA_P` is the minimum masking-induced probability drop required to count a protein as validated.
- `MIN_P_BASE` restricts the analysis to high-confidence allergen predictions.


In [3]:
K_PERCENTAGES = [0.05, 0.10, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]   # top 5%, 10%, 20%, 25%, 30%, 35%, 40%, 45%, 50% of residues
MIN_DELTA_P = 0.05                     # minimum drop to count as validated
MIN_P_BASE = 0.70                      # only high-confidence allergens, the sequence-level allergenicity model output probability must be over 0.70 to be included in the analysis


In [4]:
# --- Results refresh/reuse switch ---
REFRESH_ISM_RESULTS = True  # False: reuse CSVs and regenerate plots; True: rerun Stages 1-3 and refresh CSVs

IG_VALIDATION_SWEEP_CSV = ISM_RESULTS_DIR / "ig_validation_sweep.csv"
SATURATION_MUTAGENESIS_RESULTS_CSV = ISM_RESULTS_DIR / "saturation_mutagenesis_results.csv"
SATURATION_MUTAGENESIS_ANNOTATED_CSV = ISM_RESULTS_DIR / "saturation_mutagenesis_annotated.csv"

if not REFRESH_ISM_RESULTS:
    required_result_paths = [
        IG_VALIDATION_SWEEP_CSV,
        SATURATION_MUTAGENESIS_RESULTS_CSV,
        SATURATION_MUTAGENESIS_ANNOTATED_CSV,
    ]
    missing_result_paths = [path for path in required_result_paths if not path.exists()]
    if missing_result_paths:
        raise FileNotFoundError(
            "Cannot reuse precomputed ISM results because these files are missing: "
            + ", ".join(str(path) for path in missing_result_paths)
            + ". Set REFRESH_ISM_RESULTS=True to regenerate them."
        )

    ig_validation_sweep_df = pd.read_csv(IG_VALIDATION_SWEEP_CSV)
    saturation_mutagenesis_df = pd.read_csv(SATURATION_MUTAGENESIS_RESULTS_CSV)
    annotated_mutagenesis_df = pd.read_csv(SATURATION_MUTAGENESIS_ANNOTATED_CSV)

    expected_k_percentages = {round(float(value), 10) for value in K_PERCENTAGES}
    loaded_k_percentages = {round(float(value), 10) for value in ig_validation_sweep_df["k_pct"].unique()}
    if loaded_k_percentages != expected_k_percentages:
        missing_k = sorted(expected_k_percentages - loaded_k_percentages)
        extra_k = sorted(loaded_k_percentages - expected_k_percentages)
        print(
            "Warning: loaded ig_validation_sweep.csv does not match current K_PERCENTAGES. "
            f"Missing k values: {missing_k}; extra k values: {extra_k}. "
            "Plots will use the k values present in the loaded CSV. "
            "Set REFRESH_ISM_RESULTS=True to regenerate the sweep with current K_PERCENTAGES."
        )

    # Reconstruct best_k_pct from the sweep data using the same selection logic.
    summary_rows = []
    for k_pct, group_df in ig_validation_sweep_df.groupby("k_pct", sort=True):
        validated_values = group_df["validated"].astype(float).to_numpy()
        summary_rows.append({
            "k_pct": float(k_pct),
            "pct_validated": float(validated_values.mean()),
            "mean_delta_p": float(group_df["delta_p"].mean()),
        })
    summary_df = pd.DataFrame(summary_rows)
    best_k_row = summary_df.sort_values(
        ["pct_validated", "mean_delta_p", "k_pct"],
        ascending=[False, False, True],
    ).iloc[0]
    best_k_pct = float(best_k_row["k_pct"])
    best_k_validation_df = ig_validation_sweep_df.loc[
        (ig_validation_sweep_df["k_pct"] == best_k_pct) &
        (ig_validation_sweep_df["delta_p"] > MIN_DELTA_P)
    ].copy()

    loaded_k_label = ", ".join(f"{int(round(value * 100))}%" for value in sorted(loaded_k_percentages))
    print(
        f"Reusing precomputed ISM results. Loaded k values: {loaded_k_label}. "
        f"best_k_pct={best_k_pct:.2f}, n_validated={len(best_k_validation_df)}"
    )
else:
    print("REFRESH_ISM_RESULTS=True: recomputing Stages 1-3 and refreshing result CSVs.")


REFRESH_ISM_RESULTS=True: recomputing Stages 1-3 and refreshing result CSVs.


## Stage 1: Functional Validation Sweep

This stage loads saved raw IG vectors, sweeps across multiple percentage-based top-k thresholds, and tests whether masking those residues lowers the baseline allergenicity probability.

Output written by this stage:
- `ig_validation_sweep.csv` in `results/insilico_mutagenesis/`


In [5]:
if REFRESH_ISM_RESULTS:
    def _forward_prob(sequence: str) -> float:
        encodings = tokenizer(
            sequence,
            add_special_tokens=False,
            padding=False,
            truncation=False,
            return_tensors="pt",
        )
        encodings = {key: value.to(DEVICE) for key, value in encodings.items()}
        with torch.no_grad():
            outputs = model(encodings["input_ids"], encodings["attention_mask"])
        return float(torch.sigmoid(outputs["logits"]).item())


    def _parse_score_array(value) -> np.ndarray:
        if isinstance(value, np.ndarray):
            return value.astype(np.float32)
        if isinstance(value, list):
            return np.asarray(value, dtype=np.float32)
        if pd.isna(value):
            raise ValueError("Encountered missing IG score payload.")
        parsed = value
        if isinstance(value, str):
            try:
                parsed = json.loads(value)
            except json.JSONDecodeError:
                parsed = ast.literal_eval(value)
        arr = np.asarray(parsed, dtype=np.float32)
        if arr.ndim != 1:
            raise ValueError(f"Expected 1D IG score array, got shape {arr.shape}.")
        return arr


    raw_ig_column_candidates = [
        "ig_scores",
        "ig_scores_json",
        "integrated_gradients_scores",
        "integrated_gradients_scores_json",
    ]
    raw_ig_score_column = next((col for col in raw_ig_column_candidates if col in raw_ig_df.columns), None)
    if raw_ig_score_column is None:
        raise ValueError(
            "The baseline IG artifact does not contain raw per-residue IG scores. "
            f"Checked {RAW_IG_ROWS_PATH} for columns {raw_ig_column_candidates}. "
            "The current repo artifact only stores summary probe metrics, so stage 1 cannot proceed "
            "without a saved raw-IG export."
        )

    raw_id_column = "sequence_id" if "sequence_id" in raw_ig_df.columns else "accession"
    ig_method_mask = raw_ig_df["method"].astype(str).str.lower().eq("integrated_gradients") if "method" in raw_ig_df.columns else pd.Series(True, index=raw_ig_df.index)
    raw_ig_scores_by_id = {
        str(row[raw_id_column]): _parse_score_array(row[raw_ig_score_column])
        for _, row in raw_ig_df.loc[ig_method_mask].iterrows()
    }

    validation_rows = []
    for row in sequence_lookup_df.itertuples(index=False):
        sequence_id = str(row.sequence_id)
        sequence = str(row.sequence)
        p_base = _forward_prob(sequence)
        if p_base <= MIN_P_BASE:
            continue
        ig_scores = raw_ig_scores_by_id.get(sequence_id)
        if ig_scores is None and str(row.accession) in raw_ig_scores_by_id:
            ig_scores = raw_ig_scores_by_id[str(row.accession)]
        if ig_scores is None:
            continue
        if len(ig_scores) != len(sequence):
            raise ValueError(
                f"IG score length mismatch for {sequence_id}: len(scores)={len(ig_scores)} vs len(sequence)={len(sequence)}"
            )
        for k_pct in K_PERCENTAGES:
            result = validate_ig_residues_by_masking(
                model=model,
                tokenizer=tokenizer,
                sequence=sequence,
                ig_scores=ig_scores,
                device=DEVICE,
                k_pct=k_pct,
            )
            validation_rows.append(
                {
                    "sequence_id": sequence_id,
                    "k_pct": float(result["k_pct"]),
                    "k_absolute": int(result["k_absolute"]),
                    "p_base": float(result["p_base"]),
                    "p_masked": float(result["p_masked"]),
                    "delta_p": float(result["delta_p"]),
                    "validated": bool(result["delta_p"] > MIN_DELTA_P),
                    "top_k_indices_json": json.dumps(result["top_k_indices"]),
                }
            )

    ig_validation_sweep_df = pd.DataFrame(validation_rows)
    IG_VALIDATION_SWEEP_CSV = ISM_RESULTS_DIR / "ig_validation_sweep.csv"
    ig_validation_sweep_df.to_csv(IG_VALIDATION_SWEEP_CSV, index=False)
    print(f"Saved IG validation sweep to: {IG_VALIDATION_SWEEP_CSV}")
    ig_validation_sweep_df[["sequence_id", "k_pct", "k_absolute", "p_base", "p_masked", "delta_p", "validated"]].head()


Saved IG validation sweep to: /content/drive/MyDrive/XAllergen2.0/results/insilico_mutagenesis/ig_validation_sweep.csv


## Stage 1 Summary

This stage computes the sweep summary table in-memory so the best `k` threshold can be selected. The corresponding diagnostic plots are now rendered in `08_compare_all_model_probes.ipynb` from the saved CSVs.


In [6]:
def bootstrap_ci_mean(values, n_bootstrap: int = 1000, random_state: int = 42) -> tuple[float, float, float]:
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    if values.size == 0:
        return np.nan, np.nan, np.nan
    mean_value = float(values.mean())
    if values.size == 1:
        return mean_value, mean_value, mean_value
    rng = np.random.default_rng(random_state)
    boot = np.empty(n_bootstrap, dtype=float)
    for idx in range(n_bootstrap):
        sample = rng.choice(values, size=values.size, replace=True)
        boot[idx] = sample.mean()
    ci_low, ci_high = np.percentile(boot, [2.5, 97.5])
    return mean_value, float(ci_low), float(ci_high)

summary_rows = []
for k_pct, group_df in ig_validation_sweep_df.groupby("k_pct", sort=True):
    validated_values = group_df["validated"].astype(float).to_numpy()
    pct_validated, ci_low, ci_high = bootstrap_ci_mean(validated_values, n_bootstrap=1000, random_state=42)
    summary_rows.append(
        {
            "k_pct": float(k_pct),
            "k_absolute_mean": float(group_df["k_absolute"].mean()),
            "n_validated": int(group_df["validated"].sum()),
            "pct_validated": float(pct_validated),
            "pct_validated_ci_low": float(ci_low),
            "pct_validated_ci_high": float(ci_high),
            "n_proteins": int(len(group_df)),
            "mean_delta_p": float(group_df["delta_p"].mean()),
        }
    )
summary_df = pd.DataFrame(summary_rows).sort_values("k_pct").reset_index(drop=True)
print("Stage 1 summary computed. Final diagnostic plots now live in 08_compare_all_model_probes.ipynb.")
print(summary_df[["k_pct", "k_absolute_mean", "n_validated", "pct_validated", "mean_delta_p"]].to_string(index=False))


Stage 1 summary computed. Final diagnostic plots now live in 08_compare_all_model_probes.ipynb.
 k_pct  k_absolute_mean  n_validated  pct_validated  mean_delta_p
  0.05        14.239130           10       0.217391      0.039956
  0.10        28.086957           16       0.347826      0.085987
  0.20        55.630435           27       0.586957      0.132251
  0.25        69.304348           33       0.717391      0.188742
  0.30        83.260870           37       0.804348      0.269562
  0.35        96.978261           35       0.760870      0.362310
  0.40       110.739130           37       0.804348      0.357694
  0.45       124.630435           35       0.760870      0.336039
  0.50       138.152174           31       0.673913      0.210668


## Select the Best k Threshold

The best percentage threshold is selected by the highest validated fraction, then broken by stronger average effect size and finally by the smaller k percentage.


In [7]:
best_k_row = summary_df.sort_values(
    ["pct_validated", "mean_delta_p", "k_pct"],
    ascending=[False, False, True],
).iloc[0]
best_k_pct = float(best_k_row["k_pct"])
print(
    f"Selected best_k_pct={best_k_pct:.2f} because it has the highest validated fraction "
    f"({best_k_row['pct_validated']:.3f}); ties were broken by higher mean delta_p and then smaller k_pct."
)

best_k_validation_df = ig_validation_sweep_df.loc[
    (ig_validation_sweep_df["k_pct"] == best_k_pct) & (ig_validation_sweep_df["delta_p"] > MIN_DELTA_P)
].copy()
print(f"Validated proteins at best k: {len(best_k_validation_df)}")
best_k_validation_df.head()


Selected best_k_pct=0.40 because it has the highest validated fraction (0.804); ties were broken by higher mean delta_p and then smaller k_pct.
Validated proteins at best k: 37


,sequence_id,k_pct,k_absolute,p_base,p_masked,delta_p,validated,top_k_indices_json
6,Q84ZX5,0.4,53,0.991741,0.811469,0.180272,True,"[45, 63, 72, 49, 70, 60, 64, 76, 39, 43, 34, 2..."
15,A0ABI7XLA3,0.4,54,0.963057,0.768265,0.194792,True,"[0, 1, 83, 84, 102, 92, 87, 89, 67, 53, 88, 58..."
24,A0ABQ9K8B4,0.4,84,0.986888,0.599993,0.386895,True,"[110, 0, 115, 131, 134, 111, 107, 121, 102, 41..."
33,Q28133,0.4,69,0.933587,0.596596,0.336991,True,"[0, 1, 59, 96, 121, 31, 138, 140, 42, 92, 146,..."
42,P27762,0.4,159,0.963358,0.356767,0.606591,True,"[0, 3, 87, 22, 7, 44, 53, 1, 56, 4, 224, 70, 1..."


## Stage 1b: IG vs Random Masking Baseline

This stage writes `ig_vs_random_baseline.csv` in `results/insilico_mutagenesis/`. The paired comparison plots are now rendered in `08_compare_all_model_probes.ipynb`.


In [8]:
from scipy.stats import wilcoxon

N_RANDOM_TRIALS = 100
IG_VS_RANDOM_BASELINE_CSV = ISM_RESULTS_DIR / "ig_vs_random_baseline.csv"

def _select_best_k_pct_from_sweep(frame: pd.DataFrame) -> float:
    summary_rows = []
    for k_pct, group_df in frame.groupby("k_pct", sort=True):
        validated_values = group_df["validated"].astype(float).to_numpy()
        summary_rows.append(
            {
                "k_pct": float(k_pct),
                "pct_validated": float(validated_values.mean()),
                "mean_delta_p": float(group_df["delta_p"].mean()),
            }
        )
    summary_df_local = pd.DataFrame(summary_rows)
    best_k_row_local = summary_df_local.sort_values(["pct_validated", "mean_delta_p", "k_pct"], ascending=[False, False, True]).iloc[0]
    return float(best_k_row_local["k_pct"])

def _forward_prob_local(sequence: str) -> float:
    encodings = tokenizer(sequence, add_special_tokens=False, padding=False, truncation=False, return_tensors="pt")
    encodings = {key: value.to(DEVICE) for key, value in encodings.items()}
    with torch.no_grad():
        outputs = model(encodings["input_ids"], encodings["attention_mask"])
    return float(torch.sigmoid(outputs["logits"]).item())

if not REFRESH_ISM_RESULTS and IG_VS_RANDOM_BASELINE_CSV.exists():
    ig_vs_random_baseline_df = pd.read_csv(IG_VS_RANDOM_BASELINE_CSV)
else:
    if "ig_validation_sweep_df" not in globals():
        if "IG_VALIDATION_SWEEP_CSV" in globals() and Path(IG_VALIDATION_SWEEP_CSV).exists():
            ig_validation_sweep_df = pd.read_csv(IG_VALIDATION_SWEEP_CSV)
        else:
            raise RuntimeError("ig_validation_sweep_df is not available yet. Run Stage 1 first or set REFRESH_ISM_RESULTS=False with the precomputed sweep CSV present.")
    if "best_k_pct" not in globals():
        best_k_pct = _select_best_k_pct_from_sweep(ig_validation_sweep_df)
    best_k_all_df = ig_validation_sweep_df.loc[ig_validation_sweep_df["k_pct"] == best_k_pct].copy()
    baseline_rows = []
    for row in best_k_all_df.itertuples(index=False):
        sequence_id = str(row.sequence_id)
        sequence = sequence_lookup[sequence_id]["sequence"]
        k_absolute = int(row.k_absolute)
        p_base = float(row.p_base)
        random_delta_ps = []
        for trial_idx in range(N_RANDOM_TRIALS):
            rng = np.random.default_rng(trial_idx)
            random_indices = rng.choice(len(sequence), size=k_absolute, replace=False)
            masked_residues = list(sequence)
            for idx in random_indices:
                masked_residues[int(idx)] = tokenizer.mask_token
            p_masked_random = _forward_prob_local("".join(masked_residues))
            random_delta_ps.append(p_base - p_masked_random)
        baseline_rows.append({"sequence_id": sequence_id, "ig_delta_p": float(row.delta_p), "mean_random_delta_p": float(np.mean(random_delta_ps)), "delta_difference": float(row.delta_p) - float(np.mean(random_delta_ps)), "p_base": p_base})
    ig_vs_random_baseline_df = pd.DataFrame(baseline_rows)
    ig_vs_random_baseline_df.to_csv(IG_VS_RANDOM_BASELINE_CSV, index=False)

paired_df = ig_vs_random_baseline_df.dropna(subset=["ig_delta_p", "mean_random_delta_p"]).copy()
if paired_df.empty:
    raise ValueError("No paired IG-vs-random baseline rows are available.")
wilcoxon_result = wilcoxon(paired_df["ig_delta_p"].to_numpy(dtype=float), paired_df["mean_random_delta_p"].to_numpy(dtype=float), alternative="two-sided")
print(f"Saved IG vs random baseline CSV to: {IG_VS_RANDOM_BASELINE_CSV}")
print(f"n={len(paired_df)}, W={float(wilcoxon_result.statistic):.3f}, p-value={float(wilcoxon_result.pvalue):.3e}")
print("Diagnostic plots for this comparison now live in 08_compare_all_model_probes.ipynb.")


Saved IG vs random baseline CSV to: /content/drive/MyDrive/XAllergen2.0/results/insilico_mutagenesis/ig_vs_random_baseline.csv
n=46, W=56.000, p-value=1.832e-09
Diagnostic plots for this comparison now live in 08_compare_all_model_probes.ipynb.


## Stage 2: Saturation Mutagenesis

This stage runs single-residue substitutions at the validated positions from the best k threshold and records the mutations that reduce allergenicity probability.

Output written by this stage:
- `saturation_mutagenesis_results.csv` in `results/insilico_mutagenesis/`


In [9]:
if REFRESH_ISM_RESULTS:
    mutagenesis_frames = []
    print(f"Running saturation mutagenesis for {len(best_k_validation_df)} validated proteins at best_k_pct={best_k_pct:.2f}")
    for row in tqdm(best_k_validation_df.itertuples(index=False), total=len(best_k_validation_df), desc="Stage 2: saturation mutagenesis", unit="protein"):
        sequence_id = str(row.sequence_id)
        sequence = sequence_lookup[sequence_id]["sequence"]
        top_k_indices = json.loads(row.top_k_indices_json)
        mut_df = run_saturation_mutagenesis(
            model=model,
            tokenizer=tokenizer,
            sequence=sequence,
            target_indices=top_k_indices,
            device=DEVICE,
            p_base=float(row.p_base),
        )
        mut_df["sequence_id"] = sequence_id
        mut_df["k_pct"] = float(row.k_pct)
        mut_df["k_absolute"] = int(row.k_absolute)
        mutagenesis_frames.append(mut_df)

    saturation_mutagenesis_df = pd.concat(mutagenesis_frames, ignore_index=True) if mutagenesis_frames else pd.DataFrame(
        columns=["position", "original_aa", "mutant_aa", "p_base", "p_mutant", "delta_p", "reduces_allergenicity", "sequence_id", "k_pct", "k_absolute"]
    )
    SATURATION_MUTAGENESIS_RESULTS_CSV = ISM_RESULTS_DIR / "saturation_mutagenesis_results.csv"
    saturation_mutagenesis_df.to_csv(SATURATION_MUTAGENESIS_RESULTS_CSV, index=False)
    print(f"Saved saturation mutagenesis results to: {SATURATION_MUTAGENESIS_RESULTS_CSV}")
    saturation_mutagenesis_df.head()


Running saturation mutagenesis for 37 validated proteins at best_k_pct=0.40


Stage 2: saturation mutagenesis:   0%|          | 0/37 [00:00<?, ?protein/s]

Saved saturation mutagenesis results to: /content/drive/MyDrive/XAllergen2.0/results/insilico_mutagenesis/saturation_mutagenesis_results.csv


## Stage 3: Biochemical Annotation

Reducing substitutions are annotated with coarse amino-acid property transitions such as hydrophobic to polar or charge-preserving changes.

Output written by this stage:
- `saturation_mutagenesis_annotated.csv` in `results/insilico_mutagenesis/`


In [10]:
if REFRESH_ISM_RESULTS:
    annotated_mutagenesis_df = annotate_mutagenesis_results(saturation_mutagenesis_df)
    SATURATION_MUTAGENESIS_ANNOTATED_CSV = ISM_RESULTS_DIR / "saturation_mutagenesis_annotated.csv"
    annotated_mutagenesis_df.to_csv(SATURATION_MUTAGENESIS_ANNOTATED_CSV, index=False)
    print(f"Saved annotated mutagenesis results to: {SATURATION_MUTAGENESIS_ANNOTATED_CSV}")
    annotated_mutagenesis_df.head()


Saved annotated mutagenesis results to: /content/drive/MyDrive/XAllergen2.0/results/insilico_mutagenesis/saturation_mutagenesis_annotated.csv


## Mutagenesis Diagnostic Plotting

Mutagenesis diagnostic plots have been centralized in `08_compare_all_model_probes.ipynb`. This notebook now writes the CSV artifacts only.


In [11]:
print("Delta-p distribution plotting moved to 08_compare_all_model_probes.ipynb.")


Delta-p distribution plotting moved to 08_compare_all_model_probes.ipynb.


## Transition Analysis Summaries

This section keeps the transition-summary CSV generation in notebook 07. The corresponding figures are rendered later in `08_compare_all_model_probes.ipynb`.


In [12]:
AA_ORDER = list("ACDEFGHIKLMNPQRSTVWY")

PROPERTY_LABELS = {
    "charge_negative": "Acidic",
    "charge_positive": "Basic",
    "polar_uncharged": "Polar uncharged",
    "hydrophobic": "Nonpolar",
    "special": "Small / conformationally special",
}

def build_transition_dataframe(frame: pd.DataFrame) -> pd.DataFrame:
    required_columns = {"original_aa", "mutant_aa", "delta_p", "reduces_allergenicity", "original_property", "mutant_property"}
    missing_columns = required_columns - set(frame.columns)
    if missing_columns:
        raise ValueError(f"annotated_mutagenesis_df is missing required columns: {sorted(missing_columns)}")
    df = frame.copy()
    df = df.dropna(subset=["original_aa", "mutant_aa", "delta_p", "original_property", "mutant_property"]).copy()
    df["original_aa"] = df["original_aa"].astype(str).str.upper()
    df["mutant_aa"] = df["mutant_aa"].astype(str).str.upper()
    df = df.loc[df["original_aa"].isin(set(AA_ORDER))].copy()
    df["class"] = df["original_property"].map(PROPERTY_LABELS)
    return df.loc[df["class"].notna()].copy()

def summarize_transition_residues(frame: pd.DataFrame) -> pd.DataFrame:
    summary_rows = []
    for original_aa, group_df in frame.groupby("original_aa", sort=True):
        reducing_mask = group_df["reduces_allergenicity"].fillna(False).astype(bool)
        summary_rows.append({
            "original_aa": str(original_aa),
            "class": str(group_df["class"].mode().iloc[0]),
            "n_total": int(len(group_df)),
            "n_reducing": int(reducing_mask.sum()),
            "frac_reducing": float(reducing_mask.mean()),
            "mean_delta_p_all": float(group_df["delta_p"].mean()),
            "mean_delta_p_reducing": float(group_df.loc[reducing_mask, "delta_p"].mean()) if reducing_mask.any() else 0.0,
        })
    return pd.DataFrame(summary_rows).sort_values(["frac_reducing", "mean_delta_p_all", "original_aa"], ascending=[False, False, True]).reset_index(drop=True)

transition_df = build_transition_dataframe(annotated_mutagenesis_df)
aa_transition_summary_df = summarize_transition_residues(transition_df)
AA_TRANSITION_SUMMARY_CSV = ISM_RESULTS_DIR / "transition_panel1_residue_summary.csv"
aa_transition_summary_df.to_csv(AA_TRANSITION_SUMMARY_CSV, index=False)
print(f"Saved residue-level transition summary to: {AA_TRANSITION_SUMMARY_CSV}")


Saved residue-level transition summary to: /content/drive/MyDrive/XAllergen2.0/results/insilico_mutagenesis/transition_panel1_residue_summary.csv


In [13]:
TOP_SUPPORTED_TRANSITIONS_CSV = ISM_RESULTS_DIR / "top10_supported_aa_transitions.csv"
MIN_TRANSITION_SUPPORT = 20
BOOTSTRAP_N_RESAMPLES = 1000
BOOTSTRAP_RANDOM_STATE = 42

def bootstrap_ci_mean(values, n_bootstrap: int = BOOTSTRAP_N_RESAMPLES, random_state: int = BOOTSTRAP_RANDOM_STATE) -> tuple[float, float, float]:
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    if values.size == 0:
        return np.nan, np.nan, np.nan
    mean_value = float(values.mean())
    if values.size == 1:
        return mean_value, mean_value, mean_value
    rng = np.random.default_rng(random_state)
    boot = np.empty(n_bootstrap, dtype=float)
    for idx in range(n_bootstrap):
        sample = rng.choice(values, size=values.size, replace=True)
        boot[idx] = sample.mean()
    ci_low, ci_high = np.percentile(boot, [2.5, 97.5])
    return mean_value, float(ci_low), float(ci_high)

def summarize_top_supported_transitions(frame: pd.DataFrame, min_support: int = MIN_TRANSITION_SUPPORT, top_n: int = 10) -> pd.DataFrame:
    reducing_df = frame.copy()
    reducing_df = reducing_df.loc[reducing_df["reduces_allergenicity"].fillna(False).astype(bool)].copy()
    reducing_df = reducing_df.dropna(subset=["original_aa", "mutant_aa", "delta_p"])
    reducing_df["original_aa"] = reducing_df["original_aa"].astype(str)
    reducing_df["mutant_aa"] = reducing_df["mutant_aa"].astype(str)
    reducing_df = reducing_df.loc[reducing_df["original_aa"] != reducing_df["mutant_aa"]].copy()
    if reducing_df.empty:
        return pd.DataFrame(columns=["original_aa", "mutant_aa", "n", "mean_delta_p", "median_delta_p", "std_delta_p", "sem_delta_p", "ci95_low", "ci95_high", "ci95_half_width", "transition_label"])
    summary_df = reducing_df.groupby(["original_aa", "mutant_aa"], as_index=False)["delta_p"].agg(n="count", mean_delta_p="mean", median_delta_p="median", std_delta_p="std")
    summary_df["std_delta_p"] = summary_df["std_delta_p"].fillna(0.0)
    summary_df = summary_df.loc[summary_df["n"] >= int(min_support)].copy()
    if summary_df.empty:
        return summary_df
    summary_df["sem_delta_p"] = summary_df["std_delta_p"] / np.sqrt(summary_df["n"].clip(lower=1))
    ci_rows = []
    for row in summary_df.itertuples(index=False):
        group_values = reducing_df.loc[(reducing_df["original_aa"] == row.original_aa) & (reducing_df["mutant_aa"] == row.mutant_aa), "delta_p"].to_numpy(dtype=float)
        _, ci_low, ci_high = bootstrap_ci_mean(group_values)
        ci_rows.append({"original_aa": row.original_aa, "mutant_aa": row.mutant_aa, "ci95_low": ci_low, "ci95_high": ci_high})
    summary_df = summary_df.merge(pd.DataFrame(ci_rows), on=["original_aa", "mutant_aa"], how="left")
    summary_df["ci95_half_width"] = summary_df["ci95_high"] - summary_df["mean_delta_p"]
    summary_df["transition_label"] = summary_df["original_aa"] + "->" + summary_df["mutant_aa"]
    summary_df = summary_df.sort_values(["mean_delta_p", "n", "median_delta_p", "transition_label"], ascending=[False, False, False, True]).reset_index(drop=True)
    return summary_df.head(top_n).copy()

top_supported_transitions_df = summarize_top_supported_transitions(frame=transition_df, min_support=MIN_TRANSITION_SUPPORT, top_n=10)
top_supported_transitions_df.to_csv(TOP_SUPPORTED_TRANSITIONS_CSV, index=False)
print(f"Saved top supported transition summary to: {TOP_SUPPORTED_TRANSITIONS_CSV}")
print(top_supported_transitions_df.to_string(index=False))


Saved top supported transition summary to: /content/drive/MyDrive/XAllergen2.0/results/insilico_mutagenesis/top10_supported_aa_transitions.csv
original_aa mutant_aa   n  mean_delta_p  median_delta_p  std_delta_p  sem_delta_p  ci95_low  ci95_high  ci95_half_width transition_label
          Q         C  55      0.019176        0.000449     0.076685     0.010340  0.004565   0.045690         0.026515             Q->C
          H         K  41      0.017674        0.002856     0.040460     0.006319  0.007407   0.030503         0.012829             H->K
          L         G 122      0.015104        0.002565     0.034323     0.003107  0.009637   0.021770         0.006667             L->G
          A         M 114      0.015036        0.003139     0.062171     0.005823  0.007165   0.026895         0.011859             A->M
          Q         M 100      0.014984        0.003155     0.070112     0.007011  0.005748   0.031182         0.016198             Q->M
          S         G  40      0.01

In [14]:
print("Transition heatmap rendering moved to 08_compare_all_model_probes.ipynb.")


Transition heatmap rendering moved to 08_compare_all_model_probes.ipynb.


## Per-Protein Deep Dive CSV

This section saves a compact table for the strongest validated proteins, including the selected positions and the top reducing substitutions at each position.

Output written by this stage:
- `per_protein_deep_dive.csv` in `results/insilico_mutagenesis/`


In [15]:
PER_PROTEIN_DEEP_DIVE_CSV = ISM_RESULTS_DIR / "per_protein_deep_dive.csv"

top_validated_df = best_k_validation_df.sort_values("delta_p", ascending=False).head(10).copy()
deep_dive_rows = []
for row in top_validated_df.itertuples(index=False):
    sequence_id = str(row.sequence_id)
    top_k_indices = json.loads(row.top_k_indices_json)
    protein_mut_df = annotated_mutagenesis_df.loc[annotated_mutagenesis_df["sequence_id"] == sequence_id].copy()
    for position in top_k_indices:
        position_df = protein_mut_df.loc[
            (protein_mut_df["position"] == position) & (protein_mut_df["reduces_allergenicity"])
        ].sort_values("delta_p", ascending=False).head(3)
        for mutant_row in position_df.itertuples(index=False):
            deep_dive_rows.append(
                {
                    "sequence_id": sequence_id,
                    "masking_delta_p": float(row.delta_p),
                    "position": int(mutant_row.position),
                    "original_aa": mutant_row.original_aa,
                    "mutant_aa": mutant_row.mutant_aa,
                    "p_base": float(mutant_row.p_base),
                    "p_mutant": float(mutant_row.p_mutant),
                    "delta_p": float(mutant_row.delta_p),
                    "property_change": mutant_row.property_change,
                }
            )

per_protein_deep_dive_df = pd.DataFrame(
    deep_dive_rows,
    columns=[
        "sequence_id",
        "masking_delta_p",
        "position",
        "original_aa",
        "mutant_aa",
        "p_base",
        "p_mutant",
        "delta_p",
        "property_change",
    ],
)
per_protein_deep_dive_df = per_protein_deep_dive_df.sort_values(
    ["sequence_id", "masking_delta_p", "position", "delta_p"],
    ascending=[True, False, True, False],
).reset_index(drop=True)
per_protein_deep_dive_df.to_csv(PER_PROTEIN_DEEP_DIVE_CSV, index=False)

print(f"Saved per-protein deep dive CSV to: {PER_PROTEIN_DEEP_DIVE_CSV}")
print(
    f"Summary: proteins={top_validated_df['sequence_id'].nunique()}, "
    f"rows={len(per_protein_deep_dive_df)}"
)
print(per_protein_deep_dive_df.head().to_string(index=False))


Saved per-protein deep dive CSV to: /content/drive/MyDrive/XAllergen2.0/results/insilico_mutagenesis/per_protein_deep_dive.csv
Summary: proteins=10, rows=4053
sequence_id  masking_delta_p  position original_aa mutant_aa  p_base  p_mutant  delta_p               property_change
 A0A6P6YEX8         0.831884         0           M         W  0.9319  0.928267 0.003634                          same
 A0A6P6YEX8         0.831884         0           M         R  0.9319  0.928731 0.003170 hydrophobic → charge_positive
 A0A6P6YEX8         0.831884         0           M         K  0.9319  0.929372 0.002529 hydrophobic → charge_positive
 A0A6P6YEX8         0.831884         2           Y         R  0.9319  0.908931 0.022969 hydrophobic → charge_positive
 A0A6P6YEX8         0.831884         2           Y         K  0.9319  0.911220 0.020681 hydrophobic → charge_positive


## Output Folder

All files created by this notebook are now grouped under `results/insilico_mutagenesis/`. This makes the analysis outputs easier to inspect and keeps them separate from the general probe and training artifacts.


In [16]:
for out_path in sorted(ISM_RESULTS_DIR.glob("*")):
    print(out_path)


/content/drive/MyDrive/XAllergen2.0/results/insilico_mutagenesis/ig_validation_sweep.csv
/content/drive/MyDrive/XAllergen2.0/results/insilico_mutagenesis/ig_vs_random_baseline.csv
/content/drive/MyDrive/XAllergen2.0/results/insilico_mutagenesis/per_protein_deep_dive.csv
/content/drive/MyDrive/XAllergen2.0/results/insilico_mutagenesis/saturation_mutagenesis_annotated.csv
/content/drive/MyDrive/XAllergen2.0/results/insilico_mutagenesis/saturation_mutagenesis_results.csv
/content/drive/MyDrive/XAllergen2.0/results/insilico_mutagenesis/top10_supported_aa_transitions.csv
/content/drive/MyDrive/XAllergen2.0/results/insilico_mutagenesis/transition_panel1_residue_summary.csv
